# MLIR 编译器主线 · 第 8/8 课：Transform Dialect、Pipeline 调试与面试设计

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：把变换策略与 payload IR 分离，使用 IR dump/verify 定位第一个破坏不变量的 pass。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：编译原理基础、C++ 阅读能力
- 本课在路线中的作用：Transform dialect 用 handle 匹配 payload IR 并显式编排 tile/fuse/vectorize 等变换；它把调优策略表示成可检查 IR。

## 核心心智模型

### 1. 它是什么，解决什么问题

Transform dialect 用 handle 匹配 payload IR 并显式编排 tile/fuse/vectorize 等变换；它把调优策略表示成可检查 IR。

### 2. 它如何工作

transform op 取得 handle，应用变换后可能消费/失效旧 handle；调试时在每个 pass 后 verify，并比较第一个错误 IR。

### 3. 正确性条件与常见误区

handle 不是 payload SSA 值；payload 结构变化后旧 handle 可能失效。失败传播模式必须显式选择。

### 4. 性能与工程取舍

可调 schedule 便于 autotuning 和复现，但增加一层抽象；简单固定 pipeline 未必需要 transform dialect。

## 具体演示

先 tile linalg.matmul，再 fuse producer；若先 lower 到 loops，原 linalg handle 已不存在，后续匹配失败。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐 transform sequence 对 linalg.matmul 的匹配。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
%%writefile /tmp/lesson08.mlir
module attributes {transform.with_named_sequence} {
  transform.named_sequence @__transform_main(
      %root: !transform.any_op {transform.readonly}) {
    %matmul = transform.structured.match ops{[______]} in %root
      : (!transform.any_op) -> !transform.any_op
    transform.yield
  }
}


### 检查方法

按当前 MLIR 版本核对 transform syntax 并运行 `mlir-opt`；API 会演进，因此以仓库版本文档为准。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“Transform Dialect、Pipeline 调试与面试设计”的工作机制。

**你的答案：**


### Q2

某 pass 后 verifier 才失败，为什么应定位“第一个坏 IR”而不是最终报错 op？

**你的答案：**


### Q3

什么情况下 transform dialect 比硬编码 C++ pass pipeline 更合适？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
%%writefile /tmp/lesson08.mlir
module attributes {transform.with_named_sequence} {
  transform.named_sequence @__transform_main(
      %root: !transform.any_op {transform.readonly}) {
    %matmul = transform.structured.match ops{["linalg.matmul"]} in %root
      : (!transform.any_op) -> !transform.any_op
    transform.yield
  }
}


### Q1 参考答案

transform op 取得 handle，应用变换后可能消费/失效旧 handle；调试时在每个 pass 后 verify，并比较第一个错误 IR。

### Q2 参考答案

判断时先检查本课不变量：handle 不是 payload SSA 值；payload 结构变化后旧 handle 可能失效。失败传播模式必须显式选择。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：可调 schedule 便于 autotuning 和复现，但增加一层抽象；简单固定 pipeline 未必需要 transform dialect。

## 参考资料

- [Transform Dialect](https://mlir.llvm.org/docs/Dialects/Transform/)
- [Pattern Rewriter](https://mlir.llvm.org/docs/PatternRewriter/)
- [Dialect Conversion](https://mlir.llvm.org/docs/DialectConversion/)

资料用于建立事实基线；面试回答仍需用自己的语言组织。